# Notebook 01 — Data Collection and Physical-Statistical EDA

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

**Objective.** Build a reproducible pipeline that (i) collects hourly weather
data from Open-Meteo for 14 cities, (ii) validates the observations against
known physical constraints, (iii) performs statistically rigorous EDA, and
(iv) persists a clean snapshot in Parquet for the subsequent notebooks.

### Observational variables

| Symbol | Meaning | Unit | Physical constraint |
|---|---|---|---|
| $T_{2m}$ | 2 m air temperature | °C | $-90 \le T \le 60$ |
| $T_d$ | 2 m dewpoint | °C | $T_d \le T_{2m}$ |
| $T_{ap}$ | Apparent temperature | °C | Steadman (1979) |
| $RH$ | Relative humidity | % | $0 \le RH \le 100$ |
| $P_{msl}$ | Mean sea-level pressure | hPa | $870 \le P \le 1084$ |
| $P_s$ | Surface pressure | hPa | depends on elevation |
| $|U_{10}|$ | 10 m wind speed magnitude | km h⁻¹ | $\ge 0$ |
| $\theta_{10}$ | Wind direction (from) | ° | $[0, 360)$ |
| $G_{10}$ | 10 m wind gust | km h⁻¹ | $\ge |U_{10}|$ |
| $S^{\downarrow}$ | Shortwave radiation | W m⁻² | $\ge 0$ |
| $S^{\downarrow}_{dir}$ | Direct component | W m⁻² | $\le S^{\downarrow}$ |
| $N$ | Cloud cover | % | $[0, 100]$ |
| $P$ | Total precipitation | mm h⁻¹ | $\ge 0$ |
| WMO | Weather code (WMO 4677) | — | integer |

### Cities

14 stations across four continents, chosen to span a wide climatic spectrum:
tropical (BR, AE), temperate oceanic (UK, JP), continental (DE, US-NY),
desert (AE), subarctic/polar (NO), and humid subtropical (CN).

**Source:** [Open-Meteo Historical](https://open-meteo.com/) (no API key, free).

In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet", "csv", "json", "dtype-datetime", "rolling_window", "strings", "temporal", "rank"] }
:dep reqwest = { version = "0.12", features = ["blocking", "json"] }
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep chrono = { version = "0.4", features = ["serde"] }
:dep anyhow = "1.0"
:dep statrs = "0.18"

In [2]:
use polars::prelude::*;
use reqwest::blocking::Client;
use serde::{Deserialize, Serialize};
use std::collections::HashMap;
use std::time::Duration;
use std::thread;
use std::fs::File;

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Domain structures

We define a `City` with latitude/longitude/timezone and an initial Köppen
climate classification (tropical, arid, temperate, continental, polar) —
used only as an exploratory label, never as a model feature (this avoids
arbitrary geographic data leakage).

In [3]:
#[derive(Debug, Clone)]
struct City {
    name: String,
    country: String,
    country_code: String,
    latitude: f64,
    longitude: f64,
    elevation_m: f64,   // for surface-pressure sanity checks
    timezone: String,
    climate: &'static str,  // Köppen first letter (A/B/C/D/E)
}

impl City {
    fn new(name: &str, country: &str, code: &str, lat: f64, lon: f64,
           elev: f64, tz: &str, climate: &'static str) -> Self {
        Self {
            name: name.into(), country: country.into(), country_code: code.into(),
            latitude: lat, longitude: lon, elevation_m: elev,
            timezone: tz.into(), climate,
        }
    }
}

fn get_cities() -> Vec<City> {
    vec![
        // Brazil — tropical wet (A) and humid subtropical (Cf)
        City::new("Sao Paulo",            "Brazil", "BR", -23.55, -46.63,  760.0, "America/Sao_Paulo", "Cfa"),
        City::new("Rio de Janeiro",       "Brazil", "BR", -22.91, -43.17,    5.0, "America/Sao_Paulo", "Aw"),
        City::new("Sao Jose dos Campos",  "Brazil", "BR", -23.18, -45.88,  600.0, "America/Sao_Paulo", "Cfa"),
        City::new("Campinas",             "Brazil", "BR", -22.91, -47.06,  680.0, "America/Sao_Paulo", "Cfa"),

        // USA — continental (Df) and mediterranean (Cs)
        City::new("New York",     "USA", "US", 40.71,  -74.01,  10.0, "America/New_York",    "Dfa"),
        City::new("Los Angeles",  "USA", "US", 34.05, -118.24,  89.0, "America/Los_Angeles", "Csb"),

        // Europe — temperate oceanic (Cf) and subarctic (D/Df)
        City::new("London", "United Kingdom", "GB", 51.51,  -0.13,  35.0, "Europe/London", "Cfb"),
        City::new("Berlin", "Germany",        "DE", 52.52,  13.40,  34.0, "Europe/Berlin", "Cfb"),
        City::new("Oslo",   "Norway",         "NO", 59.91,  10.75,  23.0, "Europe/Oslo",   "Dfb"),

        // Asia — humid temperate, humid subtropical, and desert
        City::new("Tokyo",     "Japan", "JP", 35.68, 139.69,  40.0, "Asia/Tokyo",    "Cfa"),
        City::new("Shanghai",  "China", "CN", 31.23, 121.47,   4.0, "Asia/Shanghai", "Cfa"),
        City::new("Chongqing", "China", "CN", 29.56, 106.55, 244.0, "Asia/Shanghai", "Cwa"),
        City::new("Nanjing",   "China", "CN", 32.06, 118.80,  20.0, "Asia/Shanghai", "Cfa"),
        City::new("Dubai",     "UAE",   "AE", 25.27,  55.30,   5.0, "Asia/Dubai",    "BWh"),
    ]
}

let cities = get_cities();
println!("{} cities defined across {} climate zones.",
         cities.len(),
         cities.iter().map(|c| c.climate).collect::<std::collections::HashSet<_>>().len());
for (i, c) in cities.iter().enumerate() {
    println!("  {:>2}. {:<20} {:>3}  ({:>+6.2}, {:>+7.2})  elev={:>5.0} m  {}",
             i+1, c.name, c.country_code, c.latitude, c.longitude, c.elevation_m, c.climate);
}

14 cities defined across 8 climate zones.


   1. Sao Paulo             BR  (-23.55,  -46.63)  elev=  760 m  Cfa


   2. Rio de Janeiro        BR  (-22.91,  -43.17)  elev=    5 m  Aw


   3. Sao Jose dos Campos   BR  (-23.18,  -45.88)  elev=  600 m  Cfa


   4. Campinas              BR  (-22.91,  -47.06)  elev=  680 m  Cfa


...

  11. Shanghai              CN  (+31.23, +121.47)  elev=    4 m  Cfa


  12. Chongqing             CN  (+29.56, +106.55)  elev=  244 m  Cwa


  13. Nanjing               CN  (+32.06, +118.80)  elev=   20 m  Cfa


  14. Dubai                 AE  (+25.27,  +55.30)  elev=    5 m  BWh


()

---
## 2. Open-Meteo API client

The historical endpoint (`/v1/archive`) is free, requires no API key, and
returns hourly series from 1940 to the present. We implement a minimal
wrapper with a generous timeout and a manual rate limiter (500 ms) to be
courteous to the service.

In [4]:
#[derive(Debug, Deserialize)]
struct OpenMeteoResponse {
    latitude: f64,
    longitude: f64,
    #[serde(default)]
    timezone: String,
    hourly: HourlyData,
}

#[derive(Debug, Deserialize)]
struct HourlyData {
    time: Vec<String>,
    #[serde(default)] temperature_2m:        Option<Vec<Option<f64>>>,
    #[serde(default)] apparent_temperature:  Option<Vec<Option<f64>>>,
    #[serde(default)] dewpoint_2m:           Option<Vec<Option<f64>>>,
    #[serde(default)] precipitation:         Option<Vec<Option<f64>>>,
    #[serde(default)] rain:                  Option<Vec<Option<f64>>>,
    #[serde(default)] snowfall:              Option<Vec<Option<f64>>>,
    #[serde(default)] windspeed_10m:         Option<Vec<Option<f64>>>,
    #[serde(default)] windgusts_10m:         Option<Vec<Option<f64>>>,
    #[serde(default)] winddirection_10m:     Option<Vec<Option<f64>>>,
    #[serde(default)] pressure_msl:          Option<Vec<Option<f64>>>,
    #[serde(default)] surface_pressure:      Option<Vec<Option<f64>>>,
    #[serde(default)] cloudcover:            Option<Vec<Option<f64>>>,
    #[serde(default)] shortwave_radiation:   Option<Vec<Option<f64>>>,
    #[serde(default)] direct_radiation:      Option<Vec<Option<f64>>>,
    #[serde(default)] relativehumidity_2m:   Option<Vec<Option<f64>>>,
    #[serde(default)] weathercode:           Option<Vec<Option<i64>>>,
}

struct OpenMeteoClient { client: Client, archive_url: String }

impl OpenMeteoClient {
    fn new() -> Self {
        Self {
            client: Client::builder()
                .timeout(Duration::from_secs(180))
                .build().expect("HTTP client"),
            archive_url: "https://archive-api.open-meteo.com/v1/archive".to_string(),
        }
    }

    fn hourly_params() -> &'static str {
        "temperature_2m,apparent_temperature,dewpoint_2m,\
         precipitation,rain,snowfall,\
         windspeed_10m,windgusts_10m,winddirection_10m,\
         pressure_msl,surface_pressure,cloudcover,\
         shortwave_radiation,direct_radiation,\
         relativehumidity_2m,weathercode"
    }

    fn fetch_historical(&self, c: &City, start: &str, end: &str)
        -> Result<OpenMeteoResponse, Box<dyn std::error::Error>> {
        let url = format!(
            "{}?latitude={}&longitude={}&start_date={}&end_date={}&hourly={}&timezone={}",
            self.archive_url, c.latitude, c.longitude, start, end,
            Self::hourly_params(), c.timezone);
        let resp = self.client.get(&url).send()?;
        Ok(resp.json()?)
    }
}

let api_client = OpenMeteoClient::new();
println!("Open-Meteo client instantiated.");

Open-Meteo client instantiated.


---
## 3. Conversion to a Polars DataFrame

Each response is converted into a rectangular `DataFrame` holding the
observational variables plus location metadata. We use `Option<f64>` to
preserve NULLs returned by the server (they can appear even in the
historical API for very recent windows).

In [5]:
fn response_to_dataframe(c: &City, resp: &OpenMeteoResponse) -> PolarsResult<DataFrame> {
    let n = resp.hourly.time.len();
    let extract_f64 = |o: &Option<Vec<Option<f64>>>| -> Vec<Option<f64>> {
        o.as_ref().map(|v| v.clone()).unwrap_or_else(|| vec![None; n])
    };
    let extract_i64 = |o: &Option<Vec<Option<i64>>>| -> Vec<Option<i64>> {
        o.as_ref().map(|v| v.clone()).unwrap_or_else(|| vec![None; n])
    };
    let city_names: Vec<&str>     = vec![c.name.as_str(); n];
    let country_codes: Vec<&str>  = vec![c.country_code.as_str(); n];
    let climates: Vec<&str>       = vec![c.climate; n];
    let latitudes: Vec<f64>       = vec![c.latitude; n];
    let longitudes: Vec<f64>      = vec![c.longitude; n];
    let elevations: Vec<f64>      = vec![c.elevation_m; n];
    let timestamps: Vec<&str>     = resp.hourly.time.iter().map(|s| s.as_str()).collect();

    df![
        "city"                 => city_names,
        "country_code"         => country_codes,
        "climate_zone"         => climates,
        "latitude"             => latitudes,
        "longitude"            => longitudes,
        "elevation_m"          => elevations,
        "timestamp"            => timestamps,
        "temperature_2m"       => extract_f64(&resp.hourly.temperature_2m),
        "apparent_temperature" => extract_f64(&resp.hourly.apparent_temperature),
        "dewpoint_2m"          => extract_f64(&resp.hourly.dewpoint_2m),
        "precipitation"        => extract_f64(&resp.hourly.precipitation),
        "rain"                 => extract_f64(&resp.hourly.rain),
        "snowfall"             => extract_f64(&resp.hourly.snowfall),
        "windspeed_10m"        => extract_f64(&resp.hourly.windspeed_10m),
        "windgusts_10m"        => extract_f64(&resp.hourly.windgusts_10m),
        "winddirection_10m"    => extract_f64(&resp.hourly.winddirection_10m),
        "pressure_msl"         => extract_f64(&resp.hourly.pressure_msl),
        "surface_pressure"     => extract_f64(&resp.hourly.surface_pressure),
        "cloudcover"           => extract_f64(&resp.hourly.cloudcover),
        "shortwave_radiation"  => extract_f64(&resp.hourly.shortwave_radiation),
        "direct_radiation"     => extract_f64(&resp.hourly.direct_radiation),
        "relativehumidity_2m"  => extract_f64(&resp.hourly.relativehumidity_2m),
        "weathercode"          => extract_i64(&resp.hourly.weathercode)
    ]
}

println!("Conversion function ready.");

Conversion function ready.


---
## 4. Data collection

To guarantee minimum seasonal coverage — January/July cover the solar-cycle
extremes in both hemispheres — we fetch **3 representative months** of 2024
per city (January, April, July). This triples the dataset size without
inflating runtime and lets us assess seasonal trends already in the EDA.

In [6]:
// Short windows chosen to capture (i) NH winter, (ii) SH autumn and NH
// spring equinox, (iii) NH summer. Total ≈ 14 cities × 92 days × 24 h ≈ 31k rows.
let windows: Vec<(&str, &str)> = vec![
    ("2024-01-01", "2024-01-31"),
    ("2024-04-01", "2024-04-30"),
    ("2024-07-01", "2024-07-31"),
];

let mut all_dataframes: Vec<DataFrame> = Vec::new();

for (i, c) in cities.iter().enumerate() {
    for (start, end) in &windows {
        print!("[{:>2}/{:>2}] {:<20} {} -> {} ... ", i + 1, cities.len(), c.name, start, end);
        match api_client.fetch_historical(c, start, end) {
            Ok(resp) => match response_to_dataframe(c, &resp) {
                Ok(df) => { println!("ok ({} rows)", df.height()); all_dataframes.push(df); }
                Err(e) => println!("DataFrame error: {}", e),
            },
            Err(e) => println!("API error: {}", e),
        }
        thread::sleep(Duration::from_millis(400));
    }
}

println!("\nCollected {} responses.", all_dataframes.len());

[ 1/14] Sao Paulo            2024-01-01 -> 2024-01-31 ... ok (744 rows)


[ 1/14] Sao Paulo            2024-04-01 -> 2024-04-30 ... ok (720 rows)


[ 1/14] Sao Paulo            2024-07-01 -> 2024-07-31 ... ok (744 rows)


[ 2/14] Rio de Janeiro       2024-01-01 -> 2024-01-31 ... ok (744 rows)


[ 2/14] Rio de Janeiro       2024-04-01 -> 2024-04-30 ... ok (720 rows)


...

[13/14] Nanjing              2024-07-01 -> 2024-07-31 ... ok (744 rows)


[14/14] Dubai                2024-01-01 -> 2024-01-31 ... ok (744 rows)


[14/14] Dubai                2024-04-01 -> 2024-04-30 ... ok (720 rows)


[14/14] Dubai                2024-07-01 -> 2024-07-31 ... ok (744 rows)


Collected 42 responses.


In [7]:
// Stack every response into a single DataFrame.
let combined_df: DataFrame = if !all_dataframes.is_empty() {
    let mut acc = all_dataframes[0].clone();
    for df in all_dataframes.iter().skip(1) {
        acc = acc.vstack(df).expect("vstack");
    }
    acc.rechunk_mut();
    acc
} else { panic!("No DataFrame collected."); };

println!("Combined dataset: {} rows x {} columns",
         combined_df.height(), combined_df.width());
println!("Estimated memory: {:.2} MB",
         (combined_df.height() * combined_df.width() * 8) as f64 / 1_000_000.0);

Combined dataset: 30912 rows x 23 columns


Estimated memory: 5.69 MB


---
## 5. Structural sanitization

The API occasionally returns fully-null columns (for example `visibility`
from the archive endpoint). Such columns carry no information; we drop
them to stop null propagation through the rest of the pipeline.

In [8]:
// Detect and drop 100%-null columns.
let mut keep: Vec<String> = Vec::new();
let mut dropped: Vec<String> = Vec::new();
for c in combined_df.get_columns() {
    if c.null_count() == c.len() && c.len() > 0 {
        dropped.push(c.name().to_string());
    } else {
        keep.push(c.name().to_string());
    }
}
let combined_df = combined_df.select(&keep).expect("select");
if dropped.is_empty() {
    println!("No fully-null columns found.");
} else {
    println!("Dropped columns (100% null): {:?}", dropped);
}
println!("Dataset after cleaning: {} x {}", combined_df.height(), combined_df.width());

No fully-null columns found.


Dataset after cleaning: 30912 x 23


---
## 6. Physical validation

We apply first-principles constraints:

1. **Thermodynamics**: $T_d \le T_{2m}$ (the dewpoint cannot exceed the
   temperature — that would imply $RH > 100$%).
2. **Thermodynamics**: $0 \le RH \le 100$.
3. **Kinematics**: gusts $G_{10} \ge |U_{10}|$ (we tolerate a
   $\pm 0{.}5$ km/h slack due to API rounding).
4. **Radiation**: $0 \le S^{\downarrow}_{dir} \le S^{\downarrow}$.
5. **Coverage**: $0 \le N \le 100$.
6. **Pressure**: $870 \le P_{msl} \le 1084$ hPa (documented range of
   extreme meteorological events: Typhoon Tip 1979 → 870 hPa; Tosno
   anticyclone 1968 → 1084 hPa).

Each violation is counted and printed. Instead of masking violations
with nulls, we preserve the original observations so that downstream
preprocessors can decide between clip, drop, or imputation.

In [9]:
// Helper: extract a Vec<Option<f64>> from a column.
fn col_opt(df: &DataFrame, name: &str) -> Vec<Option<f64>> {
    df.column(name).unwrap().f64().unwrap().into_iter().collect()
}

let n = combined_df.height();
let t_v   = col_opt(&combined_df, "temperature_2m");
let td_v  = col_opt(&combined_df, "dewpoint_2m");
let rh_v  = col_opt(&combined_df, "relativehumidity_2m");
let g_v   = col_opt(&combined_df, "windgusts_10m");
let w_v   = col_opt(&combined_df, "windspeed_10m");
let st_v  = col_opt(&combined_df, "shortwave_radiation");
let sd_v  = col_opt(&combined_df, "direct_radiation");
let cc_v  = col_opt(&combined_df, "cloudcover");
let p_v   = col_opt(&combined_df, "pressure_msl");

let mut td_violations  = 0usize;
let mut rh_violations  = 0usize;
let mut gust_violations = 0usize;
let mut rad_violations = 0usize;
let mut cc_violations  = 0usize;
let mut p_violations   = 0usize;

for i in 0..n {
    if let (Some(t), Some(d)) = (t_v[i], td_v[i]) {
        if d > t + 1e-6 { td_violations += 1; }
    }
    if let Some(r) = rh_v[i] {
        if r < -1e-6 || r > 100.0 + 1e-6 { rh_violations += 1; }
    }
    if let (Some(g), Some(w)) = (g_v[i], w_v[i]) {
        if g + 0.5 < w { gust_violations += 1; }
    }
    if let (Some(s), Some(d)) = (st_v[i], sd_v[i]) {
        if d > s + 1e-6 { rad_violations += 1; }
    }
    if let Some(c) = cc_v[i] {
        if c < -1e-6 || c > 100.0 + 1e-6 { cc_violations += 1; }
    }
    if let Some(p) = p_v[i] {
        if p < 870.0 || p > 1084.0 { p_violations += 1; }
    }
}

println!("=== PHYSICAL VALIDATION ===");
println!("  Td > T              : {:>6}  (should be 0)", td_violations);
println!("  RH not in [0,100]   : {:>6}  (should be 0)", rh_violations);
println!("  Gust < |U|          : {:>6}  (should be 0)", gust_violations);
println!("  S_dir > S_total     : {:>6}  (should be 0)", rad_violations);
println!("  Cloudcover not in[0,100]: {:>2}  (should be 0)", cc_violations);
println!("  P_msl not in [870,1084] : {:>2}  (should be 0)", p_violations);

let total_viol = td_violations + rh_violations + gust_violations + rad_violations
                + cc_violations + p_violations;
if total_viol == 0 {
    println!("\nAll physical constraints satisfied - data is consistent.");
} else {
    println!("\n{} violations found - will be handled in Notebook 02.", total_viol);
}

=== PHYSICAL VALIDATION ===


  Td > T              :      0  (should be 0)


  RH not in [0,100]   :      0  (should be 0)


  Gust < |U|          :      2  (should be 0)


  S_dir > S_total     :      0  (should be 0)


  Cloudcover not in[0,100]:  0  (should be 0)


  P_msl not in [870,1084] :  0  (should be 0)


2 violations found - will be handled in Notebook 02.


()

---
## 7. Multivariate descriptive statistics

For each key variable we report:

- $\mu$ (mean), $\tilde{x}$ (median), $\sigma$ (standard deviation), and $IQR$.
- *Skewness* $g_1 = E\!\left[(X-\mu)^3\right]/\sigma^3$.
- Excess *kurtosis* $g_2 = E\!\left[(X-\mu)^4\right]/\sigma^4 - 3$.

These moments guide the choice of transformations (log, Box-Cox) in
Notebook 02.

In [10]:
fn moments(values: &[f64]) -> (f64, f64, f64, f64, f64, f64) {
    // Returns (mean, median, std, iqr, skewness, excess_kurtosis).
    let n = values.len() as f64;
    if n < 2.0 { return (f64::NAN, f64::NAN, f64::NAN, f64::NAN, f64::NAN, f64::NAN); }
    let mut sorted = values.to_vec();
    sorted.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let median = sorted[(n as usize)/2];
    let q1 = sorted[(n*0.25) as usize];
    let q3 = sorted[(n*0.75) as usize];
    let iqr = q3 - q1;
    let mean: f64 = values.iter().sum::<f64>() / n;
    let var: f64  = values.iter().map(|x| (x-mean).powi(2)).sum::<f64>() / (n-1.0);
    let std = var.sqrt();
    if std < 1e-12 { return (mean, median, std, iqr, 0.0, 0.0); }
    let m3: f64 = values.iter().map(|x| ((x-mean)/std).powi(3)).sum::<f64>() / n;
    let m4: f64 = values.iter().map(|x| ((x-mean)/std).powi(4)).sum::<f64>() / n;
    (mean, median, std, iqr, m3, m4 - 3.0)
}

let core_vars = [
    "temperature_2m", "dewpoint_2m", "relativehumidity_2m",
    "pressure_msl", "windspeed_10m", "cloudcover",
    "precipitation", "shortwave_radiation",
];

println!("=== DESCRIPTIVE STATISTICS (all cities) ===");
println!("{:<22} {:>9} {:>9} {:>9} {:>9} {:>9} {:>9}",
         "variable", "mean", "median", "std", "IQR", "g1", "g2-3");
println!("{}", "-".repeat(82));
for v in &core_vars {
    let s = combined_df.column(v).unwrap().f64().unwrap();
    let vec: Vec<f64> = s.into_iter().filter_map(|x| x).collect();
    let (mu, med, sd, iqr, g1, g2) = moments(&vec);
    println!("{:<22} {:>9.2} {:>9.2} {:>9.2} {:>9.2} {:>9.3} {:>9.3}",
             v, mu, med, sd, iqr, g1, g2);
}

=== DESCRIPTIVE STATISTICS (all cities) ===


variable                    mean    median       std       IQR        g1      g2-3


----------------------------------------------------------------------------------


temperature_2m             17.16     18.40     10.12     13.90    -0.501     0.176


dewpoint_2m                12.10     13.60      9.43     12.40    -0.722     0.336


...

windspeed_10m              11.34      9.90      7.04      8.90     1.261     2.574


cloudcover                 60.32     84.00     42.50     90.00    -0.385    -1.641


precipitation               0.13      0.00      0.65      0.00    11.125   173.994


shortwave_radiation       181.75     12.00    262.40    318.00     1.336     0.564


()

In [11]:
// Temperature statistics per city — useful for spotting regional outliers.
println!("=== TEMPERATURE PER CITY ===");
let temp_by_city = combined_df.clone().lazy()
    .group_by([col("city"), col("climate_zone")])
    .agg([
        col("temperature_2m").mean().alias("mean_T"),
        col("temperature_2m").median().alias("med_T"),
        col("temperature_2m").std(1).alias("std_T"),
        col("temperature_2m").min().alias("min_T"),
        col("temperature_2m").max().alias("max_T"),
        (col("temperature_2m").max() - col("temperature_2m").min()).alias("range"),
    ])
    .sort(["mean_T"], SortMultipleOptions::default().with_order_descending(true))
    .collect().unwrap();
println!("{}", temp_by_city);

=== TEMPERATURE PER CITY ===


shape: (14, 8)


┌─────────────────────┬──────────────┬───────────┬───────┬───────────┬───────┬───────┬───────┐


│ city                ┆ climate_zone ┆ mean_T    ┆ med_T ┆ std_T     ┆ min_T ┆ max_T ┆ range │


│ ---                 ┆ ---          ┆ ---       ┆ ---   ┆ ---       ┆ ---   ┆ ---   ┆ ---   │


...

│ New York            ┆ Dfa          ┆ 12.572011 ┆ 10.1  ┆ 11.167143 ┆ -8.9  ┆ 35.2  ┆ 44.1  │


│ London              ┆ Cfb          ┆ 10.863723 ┆ 10.75 ┆ 6.488555  ┆ -5.1  ┆ 29.5  ┆ 34.6  │


│ Berlin              ┆ Cfb          ┆ 10.807835 ┆ 10.4  ┆ 9.312438  ┆ -10.6 ┆ 32.2  ┆ 42.8  │


│ Oslo                ┆ Dfb          ┆ 4.937726  ┆ 5.5   ┆ 11.133839 ┆ -27.2 ┆ 25.8  ┆ 53.0  │


└─────────────────────┴──────────────┴───────────┴───────┴───────────┴───────┴───────┴───────┘


---
## 8. Robust outlier detection (per city)

We apply two criteria simultaneously, **within each city** (different
climates should not compete for the same confidence interval):

### Tukey (IQR)
$$x \in [Q_1 - 1{.}5\,IQR,\; Q_3 + 1{.}5\,IQR]$$

### MAD (Median Absolute Deviation)
$$|x - \tilde{x}| \le 3 \cdot 1{.}4826\,\text{MAD}$$

The constant $1{.}4826$ is the consistency factor for normal
distributions ($1/\Phi^{-1}(3/4)$). We count how many points would be
classified as outliers — we do not remove them here, since some of these
observations are genuine (heatwaves, cold fronts, etc.).

In [12]:
fn iqr_bounds(values: &[f64]) -> (f64, f64) {
    let mut s = values.to_vec();
    s.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let n = s.len();
    let q1 = s[n/4];
    let q3 = s[3*n/4];
    let iqr = q3 - q1;
    (q1 - 1.5*iqr, q3 + 1.5*iqr)
}

fn mad_bounds(values: &[f64]) -> (f64, f64) {
    let mut s = values.to_vec();
    s.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let med = s[s.len()/2];
    let mut dev: Vec<f64> = values.iter().map(|x| (x - med).abs()).collect();
    dev.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let mad = dev[dev.len()/2];
    let scaled = 1.4826 * mad;
    (med - 3.0*scaled, med + 3.0*scaled)
}

println!("=== PER-CITY OUTLIER DETECTION (temperature_2m) ===");
println!("{:<22} {:>9} {:>9} {:>9} {:>9} {:>10}",
         "city", "IQR_lo", "IQR_hi", "MAD_lo", "MAD_hi", "outliers");

let cities_unique: Vec<String> = combined_df.column("city").unwrap()
    .str().unwrap().into_iter()
    .filter_map(|x| x.map(String::from))
    .collect::<std::collections::HashSet<_>>()
    .into_iter().collect();

let mut outlier_summary: Vec<(String, usize)> = Vec::new();
for city_name in &cities_unique {
    let sub = combined_df.clone().lazy()
        .filter(col("city").eq(lit(city_name.clone())))
        .select([col("temperature_2m")])
        .collect().unwrap();
    let vec: Vec<f64> = sub.column("temperature_2m").unwrap().f64().unwrap()
        .into_iter().filter_map(|x| x).collect();
    if vec.is_empty() { continue; }
    let (i_lo, i_hi) = iqr_bounds(&vec);
    let (m_lo, m_hi) = mad_bounds(&vec);
    let outliers = vec.iter().filter(|x| **x < i_lo || **x > i_hi || **x < m_lo || **x > m_hi).count();
    println!("{:<22} {:>9.2} {:>9.2} {:>9.2} {:>9.2} {:>10}",
             city_name, i_lo, i_hi, m_lo, m_hi, outliers);
    outlier_summary.push((city_name.clone(), outliers));
}
let total_out: usize = outlier_summary.iter().map(|(_, n)| n).sum();
println!("\nTotal points flagged as outliers: {} ({:.2}%)",
         total_out, 100.0 * total_out as f64 / combined_df.height() as f64);

=== PER-CITY OUTLIER DETECTION (temperature_2m) ===


city                      IQR_lo    IQR_hi    MAD_lo    MAD_hi   outliers


Sao Jose dos Campos         6.85     34.45      5.68     35.92          0


Berlin                    -20.25     41.75    -24.29     45.09          0


Tokyo                     -18.15     51.85    -23.79     55.39          0


...

Rio de Janeiro             15.00     33.40     13.77     34.23         28


London                     -7.90     29.70     -9.66     31.26          0


Nanjing                   -24.15     57.85    -29.66     62.86          0


New York                  -26.60     53.40    -31.71     51.91          0


Total points flagged as outliers: 56 (0.18%)


---
## 9. Temporal autocorrelation (lag-1, lag-12, lag-24)

For a series $\{X_t\}_{t=1}^{n}$ we define
$$
\rho(k) = \frac{\sum_{t=1}^{n-k}(X_t - \bar{X})(X_{t+k} - \bar{X})}{\sum_{t=1}^{n}(X_t - \bar{X})^2}.
$$

We expect $\rho(1)$ close to 1 (hourly persistence), $\rho(24) > 0$
significant (diurnal cycle), and $\rho(12) < 0$ at mid-latitudes
(morning/afternoon anti-phase). These lags will guide feature
engineering in Notebook 02.

In [13]:
fn autocorr(values: &[f64], lag: usize) -> f64 {
    let n = values.len();
    if n <= lag + 2 { return f64::NAN; }
    let mean: f64 = values.iter().sum::<f64>() / n as f64;
    let var: f64  = values.iter().map(|x| (x-mean).powi(2)).sum::<f64>();
    if var.abs() < 1e-12 { return 0.0; }
    let cov: f64 = (0..n-lag).map(|i| (values[i]-mean)*(values[i+lag]-mean)).sum();
    cov / var
}

println!("=== PER-CITY AUTOCORRELATION (temperature_2m) ===");
println!("{:<22} {:>10} {:>10} {:>10}", "city", "rho(1)", "rho(12)", "rho(24)");
for city_name in &cities_unique {
    let sub = combined_df.clone().lazy()
        .filter(col("city").eq(lit(city_name.clone())))
        .sort(["timestamp"], Default::default())
        .select([col("temperature_2m")])
        .collect().unwrap();
    let v: Vec<f64> = sub.column("temperature_2m").unwrap().f64().unwrap()
        .into_iter().filter_map(|x| x).collect();
    if v.len() < 50 { continue; }
    println!("{:<22} {:>10.4} {:>10.4} {:>10.4}",
             city_name, autocorr(&v, 1), autocorr(&v, 12), autocorr(&v, 24));
}

=== PER-CITY AUTOCORRELATION (temperature_2m) ===


city                       rho(1)    rho(12)    rho(24)


Sao Jose dos Campos        0.9651     0.0037     0.8917


Berlin                     0.9943     0.8150     0.9269


Tokyo                      0.9941     0.8264     0.9456


...

Rio de Janeiro             0.9740     0.3535     0.8224


London                     0.9911     0.7185     0.8902


Nanjing                    0.9953     0.8777     0.9389


New York                   0.9945     0.8389     0.9315


()

---
## 10. Mean diurnal range

The daily diurnal range $\Delta T = T_{max} - T_{min}$ reveals the
thermal regime (continental vs maritime) and correlates with cloud
cover. Continental and desert climates have $\Delta T \gtrsim 15$ °C;
oceanic climates typically stay below 8 °C.

In [14]:
// Daily diurnal range (T_max - T_min per day).
// We build a 'date' column (first 10 characters of the timestamp) without
// relying on the .str() namespace -- some Polars 0.46 builds only expose it
// with the explicit 'strings' feature. We use a direct Series manipulation.
let dates: Vec<&str> = combined_df.column("timestamp").unwrap().str().unwrap()
    .into_iter().map(|opt| opt.map(|s| &s[..s.len().min(10)]).unwrap_or("")).collect();
let date_series = Series::new("date".into(), dates);
let mut df_with_date = combined_df.clone();
df_with_date.with_column(date_series).unwrap();

let diurnal = df_with_date.clone().lazy()
    .group_by([col("city"), col("date")])
    .agg([
        col("temperature_2m").max().alias("Tmax"),
        col("temperature_2m").min().alias("Tmin"),
    ])
    .with_column((col("Tmax") - col("Tmin")).alias("dT"))
    .group_by([col("city")])
    .agg([col("dT").mean().alias("mean_diurnal_range")])
    .sort(["mean_diurnal_range"], SortMultipleOptions::default().with_order_descending(true))
    .collect().unwrap();

println!("=== MEAN DIURNAL RANGE (degrees C) ===");
println!("{}", diurnal);

=== MEAN DIURNAL RANGE (degrees C) ===


shape: (14, 2)


┌─────────────────────┬────────────────────┐


│ city                ┆ mean_diurnal_range │


│ ---                 ┆ ---                │


...

│ Oslo                ┆ 7.438043           │


│ London              ┆ 7.06413            │


│ Chongqing           ┆ 6.686957           │


│ Rio de Janeiro      ┆ 6.154348           │


└─────────────────────┴────────────────────┘


---
## 11. Cross-variable correlations

Pearson is linear; in later sections (Notebook 03) we will also use
Spearman (ranks) and mutual information to capture non-linear
dependencies. Here we only need to check canonical relationships:

- $T_d \sim T$ (strong positive);
- $RH \sim T$ (moderate negative — Clausius-Clapeyron at fixed pressure);
- $S^{\downarrow} \sim N$ (strong negative);
- $|U| \sim G$ (strong positive).

In [15]:
fn pearson(a: &[f64], b: &[f64]) -> f64 {
    let n = a.len() as f64;
    if n < 3.0 { return f64::NAN; }
    let ma = a.iter().sum::<f64>() / n;
    let mb = b.iter().sum::<f64>() / n;
    let mut cov = 0.0; let mut va = 0.0; let mut vb = 0.0;
    for (x, y) in a.iter().zip(b.iter()) {
        let dx = x - ma; let dy = y - mb;
        cov += dx*dy; va += dx*dx; vb += dy*dy;
    }
    if va*vb < 1e-18 { 0.0 } else { cov / (va*vb).sqrt() }
}

let pairs = [
    ("temperature_2m", "dewpoint_2m"),
    ("temperature_2m", "relativehumidity_2m"),
    ("shortwave_radiation", "cloudcover"),
    ("windspeed_10m", "windgusts_10m"),
    ("pressure_msl", "precipitation"),
    ("temperature_2m", "apparent_temperature"),
    ("dewpoint_2m", "relativehumidity_2m"),
];

fn to_vec(df: &DataFrame, name: &str) -> Vec<f64> {
    df.column(name).unwrap().f64().unwrap()
        .into_iter().filter_map(|x| x).collect()
}

println!("=== PEARSON CORRELATIONS ===");
println!("{:<35} {:>12}", "(X, Y)", "rho");
for (a, b) in &pairs {
    let va = to_vec(&combined_df, a);
    let vb = to_vec(&combined_df, b);
    let m = va.len().min(vb.len());
    let rho = pearson(&va[..m], &vb[..m]);
    println!("{:<35} {:>12.4}", format!("({}, {})", a, b), rho);
}

=== PEARSON CORRELATIONS ===


(X, Y)                                       rho


(temperature_2m, dewpoint_2m)             0.9060


(temperature_2m, relativehumidity_2m)      -0.2797


(shortwave_radiation, cloudcover)        -0.1590


(windspeed_10m, windgusts_10m)            0.9316


(pressure_msl, precipitation)            -0.1246


(temperature_2m, apparent_temperature)       0.9892


(dewpoint_2m, relativehumidity_2m)        0.1451


()

---
## 12. Null inventory and WMO code distribution

In [16]:
println!("=== NULL INVENTORY ===");
println!("{:<25} {:>10} {:>9}", "column", "nulls", "%");
let n = combined_df.height();
for c in combined_df.get_columns() {
    let nc = c.null_count();
    if nc > 0 {
        println!("{:<25} {:>10} {:>8.2}%", c.name(), nc, 100.0 * nc as f64 / n as f64);
    }
}
println!("(Columns omitted above have 0 nulls.)");

=== NULL INVENTORY ===


column                         nulls         %


(Columns omitted above have 0 nulls.)


In [17]:
println!("=== WMO 4677 CODE DISTRIBUTION ===");
println!("Mapping:");
println!("  0-1   : clear sky / mostly clear");
println!("  2-3   : partly cloudy / overcast");
println!("  45,48 : fog");
println!("  51-67 : drizzle and rain");
println!("  71-77 : snow");
println!("  80-82 : rain showers");
println!("  85,86 : snow showers");
println!("  95-99 : thunderstorm");

let codes = combined_df.clone().lazy()
    .group_by([col("weathercode")])
    .agg([col("city").count().alias("n")])
    .sort(["n"], SortMultipleOptions::default().with_order_descending(true))
    .collect().unwrap();
println!("{}", codes);

=== WMO 4677 CODE DISTRIBUTION ===


Mapping:


  0-1   : clear sky / mostly clear


  2-3   : partly cloudy / overcast


  45,48 : fog


...

│ 71          ┆ 133   │


│ 73          ┆ 126   │


│ 65          ┆ 45    │


│ 75          ┆ 24    │


└─────────────┴───────┘


---
## 13. Persistence

We write a single clean Parquet. The CSV is no longer needed (no later
notebook consumes CSV) — this saves ~1 MB per dataset and avoids drift
between formats.

In [18]:
std::fs::create_dir_all("../data/raw").expect("mkdir");
let parquet_path = "../data/raw/weather_sample_2024_01.parquet";
let mut f = File::create(parquet_path).expect("create");
ParquetWriter::new(&mut f).finish(&mut combined_df.clone()).expect("write parquet");

println!("Saved {} ({} rows x {} columns)",
         parquet_path, combined_df.height(), combined_df.width());

Saved ../data/raw/weather_sample_2024_01.parquet (30912 rows x 23 columns)


---
## 14. Raw API fixture (for offline tests in `src/`)

We persist a single raw Open-Meteo response as a JSON fixture so that
the production code in `src/data/open_meteo.rs` can run unit tests
without network access. The fixture is committed to
`tests/fixtures/open_meteo_sample.json`.

This is part of the **equivalence contract**: any future reimplementation
of the API client must be able to deserialize this fixture and produce
the same `DataFrame` as the notebook.

In [19]:
// Fetch a fresh 2-day response for the first city and dump its raw JSON form.
// We wrap everything in a block so that `fix_city` and `fix_resp` (which hold
// references) go out of scope before the cell ends — this keeps evcxr happy.
std::fs::create_dir_all("../tests/fixtures").expect("mkdir tests/fixtures");

{
    let fix_city = cities[0].clone();
    let fix_resp = api_client.fetch_historical(&fix_city, "2024-01-01", "2024-01-02")
        .expect("fetch fixture");

    let fixture = serde_json::json!({
        "metadata": {
            "generated_from": "Notebook 01",
            "generator_version": "1.0.0",
            "purpose": "offline fixture for src/data/open_meteo.rs tests",
        },
        "city_request": {
            "name":         fix_city.name,
            "country_code": fix_city.country_code,
            "latitude":     fix_city.latitude,
            "longitude":    fix_city.longitude,
            "timezone":     fix_city.timezone,
        },
        "response": {
            "latitude":  fix_resp.latitude,
            "longitude": fix_resp.longitude,
            "timezone":  fix_resp.timezone,
            "hourly": {
                "time":                 fix_resp.hourly.time,
                "temperature_2m":       fix_resp.hourly.temperature_2m,
                "apparent_temperature": fix_resp.hourly.apparent_temperature,
                "dewpoint_2m":          fix_resp.hourly.dewpoint_2m,
                "precipitation":        fix_resp.hourly.precipitation,
                "rain":                 fix_resp.hourly.rain,
                "snowfall":             fix_resp.hourly.snowfall,
                "windspeed_10m":        fix_resp.hourly.windspeed_10m,
                "windgusts_10m":        fix_resp.hourly.windgusts_10m,
                "winddirection_10m":    fix_resp.hourly.winddirection_10m,
                "pressure_msl":         fix_resp.hourly.pressure_msl,
                "surface_pressure":     fix_resp.hourly.surface_pressure,
                "cloudcover":           fix_resp.hourly.cloudcover,
                "shortwave_radiation":  fix_resp.hourly.shortwave_radiation,
                "direct_radiation":     fix_resp.hourly.direct_radiation,
                "relativehumidity_2m":  fix_resp.hourly.relativehumidity_2m,
                "weathercode":          fix_resp.hourly.weathercode,
            }
        }
    });

    std::fs::write("../tests/fixtures/open_meteo_sample.json",
        serde_json::to_string_pretty(&fixture).unwrap()).unwrap();
    println!("Saved ../tests/fixtures/open_meteo_sample.json ({} hours)",
        fix_resp.hourly.time.len());
}

Saved ../tests/fixtures/open_meteo_sample.json (48 hours)


()

---
## 15. Summary

| step | result |
|---|---|
| Collection | 14 cities x 3 windows x 24 h ~ 31k hourly rows |
| Sanitization | 100%-null columns dropped automatically |
| Physical validation | 6 invariants checked |
| Statistics | full moments per variable |
| Outliers | per-city IQR + MAD |
| Autocorrelation | rho(1), rho(12), rho(24) per city |
| Persistence | single Parquet in `data/raw/` |
| Fixture | `tests/fixtures/open_meteo_sample.json` |

-> Next: **Notebook 02** — preprocessing and physical feature engineering.

In [20]:
println!("\n{}", "=".repeat(60));
println!("Notebook 01 complete.");
println!("{}", "=".repeat(60));
println!("Next: Notebook 02 - Preprocessing and Feature Engineering");

Notebook 01 complete.


Next: Notebook 02 - Preprocessing and Feature Engineering
